# Funding vs return exploration with BTCDOM gating

Load joined MSM timeseries (with `btcdom_recon`), compute correlations, and explore implementable strategies: gate by BTCDOM, size by BTCDOM, label + BTCDOM.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path to joined timeseries (run join_btcdom.py first)
DATA_PATH = Path("out/msm_timeseries_with_btcdom.csv")  # adjust if needed
df = pd.read_csv(DATA_PATH)
df["decision_date"] = pd.to_datetime(df["decision_date"])
df = df.dropna(subset=["F_tk", "y"])
print(f"Rows: {len(df)}, cols: {list(df.columns)}")

## 1. Overall correlation: F_tk vs y

In [ ]:
r_pearson = df["F_tk"].corr(df["y"])
r_spearman = df["F_tk"].corr(df["y"], method="spearman")
print(f"Pearson:  {r_pearson:.4f}")
print(f"Spearman: {r_spearman:.4f}")
print(f"mean(y):  {df['y'].mean():.4f}, hit_rate(y<0): {(df['y'] < 0).mean():.2%}")

## 2. By BTCDOM regime (quartiles)

In [ ]:
if "btcdom_recon" in df.columns and df["btcdom_recon"].notna().any():
    df["btcdom_regime"] = pd.qcut(df["btcdom_recon"].rank(method="first"), 4, labels=[1, 2, 3, 4])
    for r in [1, 2, 3, 4]:
        sub = df[df["btcdom_regime"] == r]
        if len(sub) < 3:
            continue
        corr = sub["F_tk"].corr(sub["y"])
        print(f"Regime {r}: n={len(sub)} pearson={corr:.4f} mean_y={sub['y'].mean():.4f}")
else:
    print("No btcdom_recon in data; run join_btcdom.py first.")

## 3. Strategy: Gate by BTCDOM (only trade when BTCDOM in range)

In [ ]:
if "btcdom_recon" not in df.columns or df["btcdom_recon"].isna().all():
    print("Skip: no btcdom_recon")
else:
    # Always-on: full sample
    cum_always = (1 + df["y"]).cumprod()
    ret_always = cum_always.iloc[-1] - 1 if len(cum_always) else 0
    hit_always = (df["y"] > 0).mean()
    print(f"Always-on: cum_return={ret_always:.4f} hit_rate={hit_always:.2%} n={len(df)}")
    # Gate: e.g. only when BTCDOM below 5000 (risk-on alts)
    gate = df["btcdom_recon"] < 5000
    gated = df[gate]
    if len(gated) >= 3:
        cum_gate = (1 + gated["y"]).cumprod()
        ret_gate = cum_gate.iloc[-1] - 1
        hit_gate = (gated["y"] > 0).mean()
        print(f"Gate BTCDOM<5000: cum_return={ret_gate:.4f} hit_rate={hit_gate:.2%} n={len(gated)}")
    # Gate: BTCDOM between 4000 and 5500
    gate2 = (df["btcdom_recon"] >= 4000) & (df["btcdom_recon"] <= 5500)
    gated2 = df[gate2]
    if len(gated2) >= 3:
        cum2 = (1 + gated2["y"]).cumprod()
        print(f"Gate 4000<=BTCDOM<=5500: cum_return={cum2.iloc[-1]-1:.4f} hit_rate={(gated2['y']>0).mean():.2%} n={len(gated2)}")

## 4. Strategy: Size by BTCDOM (scale exposure)

In [ ]:
if "btcdom_recon" not in df.columns or df["btcdom_recon"].isna().all():
    print("Skip: no btcdom_recon")
else:
    # Scale: 0 when BTCDOM > 5000, 1 when < 4000, linear between
    b = df["btcdom_recon"]
    scale = np.clip((5000 - b) / 1000, 0, 1)
    scaled_ret = (df["y"] * scale).values
    cum_scaled = np.cumprod(1 + scaled_ret)
    print(f"Size by BTCDOM (0 above 5k, 1 below 4k): cum_return={cum_scaled[-1]-1:.4f}")

## 5. Strategy: Label + BTCDOM (trade only when Green and BTCDOM in range)

In [ ]:
if "label_v0_0" not in df.columns or "btcdom_recon" not in df.columns:
    print("Skip: need label_v0_0 and btcdom_recon")
else:
    green = df["label_v0_0"] == "Green"
    btc_band = (df["btcdom_recon"] >= 4000) & (df["btcdom_recon"] <= 5500)
    sel = green & btc_band
    sub = df[sel]
    if len(sub) >= 3:
        cum = (1 + sub["y"]).cumprod()
        print(f"Green & 4000<=BTCDOM<=5500: n={len(sub)} cum_return={cum.iloc[-1]-1:.4f} hit_rate={(sub['y']>0).mean():.2%}")
    else:
        print(f"Green & BTCDOM band: n={len(sub)} (too few)")

## 6. Findings and strategies (summary)

- **Strength of relationship**: Inspect Pearson/Spearman above; if weak, funding may have limited predictive power for forward LS return.
- **BTCDOM gating**: Compare always-on vs gated cumulative return and hit rate; if gating improves risk/return, use threshold (e.g. only trade when BTCDOM &lt; 5000 or in [4000, 5500]).
- **Implementable ideas**: (1) **Gate**: take LS exposure only when BTCDOM in chosen band. (2) **Size by BTCDOM**: scale notional by a function of BTCDOM (e.g. zero above 5k, full below 4k). (3) **Label + BTCDOM**: only trade when funding label is Green and BTCDOM in range; tune band and label definition.